# Análise histórica dos custodiantes de fundos da CVM

**Contexto.** O cadastro histórico de fundos de investimento da CVM registra, para cada fundo, os custodiantes que o atenderam e os períodos de cada relação. Este notebook reconstrói essa história mês a mês.

**Perguntas**
1. Quantos custodiantes diferentes existem no histórico da base?
2. Quantos custodiantes estavam ativos ao longo do tempo?
3. Quantos fundos cada custodiante mantinha sob custódia em cada período?
4. Como evoluiu a distribuição dos fundos entre os custodiantes?
5. Quais custodiantes tiveram crescimento ou redução relevante?

## 1. Fonte e obtenção dos dados

Fonte oficial: <https://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi_hist.zip>. A cada execução o ZIP é baixado, validado (status HTTP, tamanho, integridade) e o CSV de custodiantes é localizado pelo nome dentro do arquivo. A cópia local `cad_fi_hist_custodiante.csv` é sobrescrita. **Se o download falhar, a análise é interrompida**: nenhuma cópia antiga é usada em silêncio.

## 2. Importações e configurações

In [1]:
from pathlib import Path
import json, re
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

import custodiantes as cu
import graficos as gr

RAIZ = Path.cwd()
CSV_LOCAL = RAIZ / "cad_fi_hist_custodiante.csv"
SAIDA = RAIZ / "resultados"
SAIDA.mkdir(exist_ok=True)
README = RAIZ / "README.md"

TOP_N_VISIVEIS = 10          # linhas visíveis por padrão no gráfico principal
JANELA_MESES = 12            # janela da comparação de crescimento/redução
LIMIAR_VARIACAO = 100        # variação absoluta (fundos) considerada relevante
pd.options.display.float_format = "{:,.1f}".format

## 3. Carregamento e validação

In [2]:
bruto, meta = cu.atualizar_base(CSV_LOCAL)
DATA_EXTRACAO = meta["data_extracao"]
print(f"Download OK: {meta['bytes']:,} bytes | Last-Modified: {meta['last_modified']}")
print(f"CSV no ZIP: {meta['csv_no_zip']} | encoding: {meta['encoding']} | delimitador: {meta['delimitador']!r}")
print(f"Data da extração: {DATA_EXTRACAO:%Y-%m-%d} | linhas brutas: {len(bruto):,}")
assert list(bruto.columns) == cu.COLUNAS
bruto.head()

Download OK: 18,705,346 bytes | Last-Modified: Wed, 21 Oct 2026 04:17:53 GMT
CSV no ZIP: cad_fi_hist_custodiante.csv | encoding: cp1252 | delimitador: ';'
Data da extração: 2026-10-21 | linhas brutas: 73,005


,CNPJ_FUNDO,DT_REG,CNPJ_CUSTODIANTE,CUSTODIANTE,DT_INI_CUSTODIANTE,DT_FIM_CUSTODIANTE
0,00.000.432/0001-00,2005-03-31,61.472.676/0001-72,BANCO SANTANDER BRASIL SA,2005-03-31,2006-11-22
1,00.000.432/0001-00,2005-03-31,90.400.888/0001-42,BANCO SANTANDER (BRASIL) S.A.,2006-11-22,2008-07-18
2,00.000.744/0001-06,2005-01-13,33.066.408/0001-15,BANCO ABN AMRO REAL S.A.,2002-05-13,2006-06-30
3,00.000.746/0001-03,2004-12-09,62.178.421/0001-64,ALFA CORRETORA DE CAMBIO E VALORES MOBILIARIOS...,2004-03-05,2008-11-17
4,00.000.746/0001-03,2004-12-09,60.746.948/0001-12,BANCO BRADESCO S.A.,2008-11-17,


## 4. Tratamento dos dados

- `CNPJ_FUNDO` e `CNPJ_CUSTODIANTE` permanecem como texto; datas são convertidas (`DT_FIM_CUSTODIANTE` vazia = sem encerramento).
- São descartadas apenas linhas sem CNPJ, sem `DT_INI_CUSTODIANTE` válida (vigência indeterminável) ou com `DT_FIM < DT_INI`; duplicatas **exatas** são removidas. Mudanças de custodiante (linhas distintas) são preservadas.
- Cada descarte é contabilizado na tabela abaixo.

In [3]:
dados, qualidade = cu.preparar(bruto)
qualidade.to_csv(SAIDA / "qualidade_dados.csv", index=False)
display(qualidade)

,verificacao,linhas,tratamento
0,Linhas lidas,73005,—
1,DT_REG: valor preenchido porém inválido,0,tratada como ausente
2,DT_INI_CUSTODIANTE: valor preenchido porém inv...,0,tratada como ausente (descartada se for DT_INI...
3,DT_FIM_CUSTODIANTE: valor preenchido porém inv...,0,tratada como ausente
4,DT_FIM_CUSTODIANTE vazia (relação vigente),41531,mantida; vigente até a data de extração
5,CNPJ_FUNDO ausente,0,linha descartada
6,CNPJ_CUSTODIANTE ausente,0,linha descartada
7,DT_INI_CUSTODIANTE ausente/inválida,0,linha descartada (vigência indeterminável)
8,DT_FIM < DT_INI (intervalo invertido),0,linha descartada
9,DT_FIM = DT_INI (vigência de duração zero),0,mantida; nunca é contada no intervalo semiaberto


## 5. Construção da série histórica

- **Vigência (intervalo semiaberto):** `DT_INI_CUSTODIANTE <= data < DT_FIM_CUSTODIANTE`. Assim, numa transferência com fim do antigo e início do novo na mesma data, só o novo conta. Relação sem data final é vigente até a data da extração (inclusive). `DT_REG` não é usado como data de custódia.
- **Datas de referência:** último dia de cada mês, do mês da primeira relação até a extração, mais a própria data da extração (se não for fim de mês). Nada posterior à extração.
- **Contagem:** fundos *distintos* por custodiante (intervalos sobrepostos do mesmo fundo/custodiante são unidos antes de contar).
- Os resultados refletem o arquivo da extração atual, não versões anteriores publicadas pela CVM.

In [4]:
serie, totais = cu.serie_historica(dados, DATA_EXTRACAO)
assert serie["DATA_REFERENCIA"].max() == DATA_EXTRACAO == totais["DATA_REFERENCIA"].max()
print(f"{totais['DATA_REFERENCIA'].min():%Y-%m-%d} a {totais['DATA_REFERENCIA'].max():%Y-%m-%d}: "
      f"{len(totais)} datas de referência, {len(serie):,} linhas na tabela longa")

1990-08-31 a 2026-10-21: 435 datas de referência, 18,841 linhas na tabela longa


## 6. Indicadores e agregações

In [5]:
# Consistência: soma por custodiante >= total distinto (um fundo pode constar em 2 custodiantes na mesma data)
soma = serie.groupby("DATA_REFERENCIA")["QTD_FUNDOS"].sum()
assert (soma.reindex(totais["DATA_REFERENCIA"]).to_numpy() >= totais["QTD_FUNDOS_DISTINTOS_TOTAL"].to_numpy()).all()
assert (serie.groupby("DATA_REFERENCIA")["CNPJ_CUSTODIANTE"].nunique().reindex(totais["DATA_REFERENCIA"]).to_numpy()
        == totais["QTD_CUSTODIANTES_ATIVOS"].to_numpy()).all()

ultima = totais["DATA_REFERENCIA"].max()
ult = totais.set_index("DATA_REFERENCIA").loc[ultima]
resumo = {
    "data_extracao": f"{DATA_EXTRACAO:%Y-%m-%d}",
    "last_modified_http": meta["last_modified"],
    "registros_brutos": int(len(bruto)),
    "registros_analisados": int(len(dados)),
    "fundos_distintos_historico": int(dados["CNPJ_FUNDO"].nunique()),
    "custodiantes_distintos_historico": int(dados["CNPJ_CUSTODIANTE"].nunique()),
    "primeira_data_referencia": f"{totais['DATA_REFERENCIA'].min():%Y-%m-%d}",
    "ultima_data_referencia": f"{ultima:%Y-%m-%d}",
    "custodiantes_ativos_ultima_data": int(ult["QTD_CUSTODIANTES_ATIVOS"]),
    "fundos_distintos_sob_custodia_ultima_data": int(ult["QTD_FUNDOS_DISTINTOS_TOTAL"]),
    "pico_custodiantes_ativos": int(totais["QTD_CUSTODIANTES_ATIVOS"].max()),
    "data_pico_custodiantes_ativos": f"{totais.loc[totais['QTD_CUSTODIANTES_ATIVOS'].idxmax(), 'DATA_REFERENCIA']:%Y-%m-%d}",
}
display(pd.Series(resumo, name="valor").to_frame())

atual = serie[serie["DATA_REFERENCIA"] == ultima].sort_values("QTD_FUNDOS", ascending=False).copy()
atual["PARTICIPACAO_%"] = 100 * atual["QTD_FUNDOS"] / atual["QTD_FUNDOS"].sum()
display(atual.head(15).reset_index(drop=True))

,valor
data_extracao,2026-10-21
last_modified_http,"Wed, 21 Oct 2026 04:17:53 GMT"
registros_brutos,73005
registros_analisados,72989
fundos_distintos_historico,54933
custodiantes_distintos_historico,118
primeira_data_referencia,1990-08-31
ultima_data_referencia,2026-10-21
custodiantes_ativos_ultima_data,95
fundos_distintos_sob_custodia_ultima_data,41349


,DATA_REFERENCIA,CNPJ_CUSTODIANTE,CUSTODIANTE,QTD_FUNDOS,PARTICIPACAO_%
0,2026-10-21,60.701.190/0001-04,ITAU UNIBANCO S.A.,10413,25.1
1,2026-10-21,30.306.294/0001-45,BANCO BTG PACTUAL S/A,7546,18.2
2,2026-10-21,60.746.948/0001-12,BANCO BRADESCO S.A.,7325,17.7
3,2026-10-21,62.318.407/0001-19,S3 CACEIS BRASIL DISTRIBUIDORA DE TITULOS E VA...,3813,9.2
4,2026-10-21,42.272.526/0001-70,BNY MELLON BANCO S.A.,3509,8.5
5,2026-10-21,58.160.789/0001-28,BANCO SAFRA S/A,1997,4.8
6,2026-10-21,00.000.000/0001-91,BANCO DO BRASIL S.A.,1620,3.9
7,2026-10-21,62.232.889/0001-90,BANCO DAYCOVAL S.A.,561,1.4
8,2026-10-21,01.522.368/0001-82,BANCO BNP PARIBAS BRASIL S/A,505,1.2
9,2026-10-21,62.285.390/0001-40,QI CORRETORA DE TÍTULOS E VALORES MOBILIÁRIOS ...,396,1.0


## 7. Visualizações

In [6]:
nomes = cu.nomes_exibicao(dados)
# Use os controles acima do gráfico: caixa de seleção de custodiantes (o restante vira "Outros") e intervalo de datas.
dados_g1 = gr.montar_dados(
    serie, nomes, top_n=TOP_N_VISIVEIS, eixo_y="Fundos distintos sob custódia",
    titulo=f"Fundos sob custódia por custodiante (mensal, até {ultima:%d/%m/%Y})")
doc_g1 = gr.salvar(dados_g1, SAIDA / "grafico_fundos_por_custodiante.html")
display(HTML(gr.iframe(doc_g1)))

C:\Users\bcasanovas\Pyhton Files\dash_custodiante_hist\.venv\Lib\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Market share ao longo do tempo

Mesma lógica do gráfico anterior (seleção de custodiantes, "Outros" e intervalo de datas), mas exibindo a **participação percentual** de cada custodiante no total de vínculos fundo-custodiante da data. Um fundo com dois custodiantes simultâneos conta uma vez em cada, então o denominador é a soma das contagens por custodiante (não o total de fundos distintos).

In [7]:
dados_g_share = gr.montar_dados(
    serie, nomes, top_n=TOP_N_VISIVEIS, modo="share", eixo_y="Participação nos fundos sob custódia (%)",
    titulo=f"Market share por custodiante (% dos vínculos fundo-custodiante, até {ultima:%d/%m/%Y})")
doc_g_share = gr.salvar(dados_g_share, SAIDA / "grafico_market_share_custodiantes.html")
display(HTML(gr.iframe(doc_g_share)))

C:\Users\bcasanovas\Pyhton Files\dash_custodiante_hist\.venv\Lib\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [8]:
fig2 = go.Figure(go.Scatter(x=totais["DATA_REFERENCIA"], y=totais["QTD_CUSTODIANTES_ATIVOS"], mode="lines",
                            name="Custodiantes ativos", hovertemplate="%{x|%d/%m/%Y}: %{y} custodiantes<extra></extra>"))
fig2.update_layout(title="Custodiantes ativos (≥ 1 fundo sob custódia) por data de referência",
                   xaxis_title="Data de referência", yaxis_title="Quantidade de custodiantes ativos",
                   height=450, xaxis_tickformat="%b/%Y")
fig2.write_html(SAIDA / "grafico_custodiantes_ativos.html", include_plotlyjs="cdn")
fig2.show()

In [9]:
# Distribuição: participação dos 5 maiores de cada data e índice de Herfindahl-Hirschman (HHI, 0-10.000)
def concentracao(g):
    p = 100 * g["QTD_FUNDOS"] / g["QTD_FUNDOS"].sum()
    return pd.Series({"PART_TOP5_%": p.nlargest(5).sum(), "HHI": (p ** 2).sum()})
conc = serie.groupby("DATA_REFERENCIA").apply(concentracao, include_groups=False).reset_index()
conc.to_csv(SAIDA / "concentracao_por_data.csv", index=False)

fig3 = go.Figure(go.Scatter(x=conc["DATA_REFERENCIA"], y=conc["PART_TOP5_%"], mode="lines", name="Top 5"))
fig3.update_layout(title="Participação dos 5 maiores custodiantes no total de vínculos de custódia (%)",
                   xaxis_title="Data de referência", yaxis_title="% dos vínculos fundo-custodiante", height=400)
fig3.show()

## 8. Análise dos resultados

In [10]:
piv = serie.pivot_table(index="CNPJ_CUSTODIANTE", columns="DATA_REFERENCIA", values="QTD_FUNDOS", fill_value=0)
ini_janela = ultima - pd.DateOffset(months=JANELA_MESES)
col_ini = piv.columns[piv.columns <= ini_janela].max()
var = pd.DataFrame({"CUSTODIANTE": nomes.reindex(piv.index),
                    f"QTD_{col_ini:%Y-%m-%d}": piv[col_ini], f"QTD_{ultima:%Y-%m-%d}": piv[ultima],
                    "PICO": piv.max(axis=1), "DATA_PICO": piv.idxmax(axis=1)})
var["VARIACAO"] = var.iloc[:, 2] - var.iloc[:, 1]
var["VARIACAO_%"] = 100 * var["VARIACAO"] / var.iloc[:, 1].where(var.iloc[:, 1] > 0)
var["QUEDA_DESDE_PICO"] = var["PICO"] - var.iloc[:, 2]
var = var.sort_values("VARIACAO", ascending=False)
var.reset_index().to_csv(SAIDA / "variacao_custodiantes.csv", index=False)

display(Markdown(f"**Variação em {JANELA_MESES} meses ({col_ini:%d/%m/%Y} → {ultima:%d/%m/%Y}); relevante: |variação| ≥ {LIMIAR_VARIACAO} fundos**"))
cresc = var[var["VARIACAO"] >= LIMIAR_VARIACAO]
reduc = var[var["VARIACAO"] <= -LIMIAR_VARIACAO].sort_values("VARIACAO")
display(Markdown(f"Crescimento relevante: {len(cresc)} custodiante(s)")); display(cresc.head(10))
display(Markdown(f"Redução relevante: {len(reduc)} custodiante(s)")); display(reduc.head(10))
display(Markdown("**Maiores quedas desde o próprio pico**")); display(var.sort_values("QUEDA_DESDE_PICO", ascending=False).head(10))

anos = conc[conc["DATA_REFERENCIA"].dt.month.eq(12) | conc["DATA_REFERENCIA"].eq(ultima)]
display(Markdown("**Concentração (fim de cada ano e última data)**")); display(anos.set_index("DATA_REFERENCIA"))

**Variação em 12 meses (30/09/2025 → 21/10/2026); relevante: |variação| ≥ 100 fundos**

Crescimento relevante: 0 custodiante(s)

,CUSTODIANTE,QTD_2025-09-30,QTD_2026-10-21,PICO,DATA_PICO,VARIACAO,VARIACAO_%,QUEDA_DESDE_PICO
CNPJ_CUSTODIANTE,,,,,,,,


Redução relevante: 0 custodiante(s)

,CUSTODIANTE,QTD_2025-09-30,QTD_2026-10-21,PICO,DATA_PICO,VARIACAO,VARIACAO_%,QUEDA_DESDE_PICO
CNPJ_CUSTODIANTE,,,,,,,,


**Maiores quedas desde o próprio pico**

,CUSTODIANTE,QTD_2025-09-30,QTD_2026-10-21,PICO,DATA_PICO,VARIACAO,VARIACAO_%,QUEDA_DESDE_PICO
CNPJ_CUSTODIANTE,,,,,,,,
33.868.597/0001-40,CITIBANK DTVM SA,198.0,198.0,"1,262.0",2013-02-28,0.0,0.0,"1,064.0"
90.400.888/0001-42,BANCO SANTANDER (BRASIL) S.A.,247.0,247.0,985.0,2014-01-31,0.0,0.0,738.0
33.700.394/0001-40,UNIBANCO UNIAO BANCOS BRAS SA,80.0,80.0,638.0,2007-12-31,0.0,0.0,558.0
30.822.936/0001-69,BB GESTAO DE RECURSOS DTVM S.A,49.0,49.0,480.0,2013-08-31,0.0,0.0,431.0
01.701.201/0001-89,KIRTON BANK S.A. - BANCO MÚLTIPLO,111.0,111.0,536.0,2011-03-31,0.0,0.0,425.0
61.472.676/0001-72,BANCO SANTANDER BRASIL SA,7.0,7.0,340.0,2006-04-30,0.0,0.0,333.0
58.229.246/0001-10,BRADESCO-KIRTON CORRETORA DE CÂMBIO S.A.,36.0,36.0,319.0,2005-03-31,0.0,0.0,283.0
33.066.408/0001-15,BANCO ABN AMRO REAL S.A.,74.0,74.0,307.0,2007-09-30,0.0,0.0,233.0
42.272.526/0001-70,BNY MELLON BANCO S.A.,"3,509.0","3,509.0","3,735.0",2024-05-31,0.0,0.0,226.0


**Concentração (fim de cada ano e última data)**

,PART_TOP5_%,HHI
DATA_REFERENCIA,,
1990-12-31,100.0,"10,000.0"
1991-12-31,100.0,"5,000.0"
1992-12-31,100.0,"5,200.0"
1993-12-31,100.0,"5,061.7"
1994-12-31,100.0,"5,029.6"
1995-12-31,100.0,"4,433.6"
1996-12-31,100.0,"3,447.9"
1997-12-31,100.0,"3,424.5"
1998-12-31,92.8,"2,357.6"


In [11]:
top1 = atual.iloc[0]
display(Markdown(f'''
- O histórico contém **{resumo['custodiantes_distintos_historico']}** custodiantes distintos (por CNPJ); em {ultima:%d/%m/%Y}, **{resumo['custodiantes_ativos_ultima_data']}** estavam ativos (pico de {resumo['pico_custodiantes_ativos']} em {resumo['data_pico_custodiantes_ativos']}).
- Em {ultima:%d/%m/%Y} havia **{resumo['fundos_distintos_sob_custodia_ultima_data']:,}** fundos distintos com custódia vigente; o maior custodiante é **{top1['CUSTODIANTE']}** ({top1['QTD_FUNDOS']:,} fundos, {top1['PARTICIPACAO_%']:.1f}% dos vínculos).
- Os 5 maiores concentravam {conc['PART_TOP5_%'].iloc[-1]:.1f}% dos vínculos na última data (HHI {conc['HHI'].iloc[-1]:,.0f}), contra {conc['PART_TOP5_%'].iloc[0]:.1f}% na primeira data da série.
- Observações descritivas; este notebook não atribui causas às variações.
'''))


- O histórico contém **118** custodiantes distintos (por CNPJ); em 21/10/2026, **95** estavam ativos (pico de 95 em 2024-11-30).
- Em 21/10/2026 havia **41,349** fundos distintos com custódia vigente; o maior custodiante é **ITAU UNIBANCO S.A.** (10,413 fundos, 25.1% dos vínculos).
- Os 5 maiores concentravam 78.6% dos vínculos na última data (HHI 1,474), contra 100.0% na primeira data da série.
- Observações descritivas; este notebook não atribui causas às variações.


## 9. Exportação e atualização do README

In [12]:
serie.to_csv(SAIDA / "custodiantes_serie_historica.csv", index=False)
totais.to_csv(SAIDA / "totais_por_data.csv", index=False)
(SAIDA / "resumo_indicadores.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8")

# Recarrega o CSV exportado e confere contra os dados em memória (tabela agregada = gráfico)
chk = pd.read_csv(SAIDA / "custodiantes_serie_historica.csv", dtype={"CNPJ_CUSTODIANTE": str}, parse_dates=["DATA_REFERENCIA"])
assert chk["QTD_FUNDOS"].sum() == serie["QTD_FUNDOS"].sum()
assert int(sum(sum(x["v"]) for x in dados_g1["series"])) == int(serie["QTD_FUNDOS"].sum())
soma_share = pd.DataFrame([x["v"] for x in dados_g_share["series"]]).sum()
assert ((soma_share[soma_share > 0] - 100).abs() < 0.01).all(), "o share de cada data deve somar 100%"
assert int(sum(fig2.data[0].y)) == int(totais["QTD_CUSTODIANTES_ATIVOS"].sum())

def tabela_md(df):
    return "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n" + \
           "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False))

top = atual.head(10)[["CUSTODIANTE", "CNPJ_CUSTODIANTE", "QTD_FUNDOS"]].copy()
top["QTD_FUNDOS"] = top["QTD_FUNDOS"].map("{:,}".format)
bloco = f'''<!-- RESULTADOS:INICIO -->
**Última extração bem-sucedida:** {DATA_EXTRACAO:%d/%m/%Y} (cabeçalho HTTP `Last-Modified`: {meta['last_modified']})

| Indicador | Valor |
|---|---|
| Registros brutos / analisados | {resumo['registros_brutos']:,} / {resumo['registros_analisados']:,} |
| Fundos distintos no histórico | {resumo['fundos_distintos_historico']:,} |
| Custodiantes distintos no histórico | {resumo['custodiantes_distintos_historico']} |
| Série | {resumo['primeira_data_referencia']} a {resumo['ultima_data_referencia']} ({len(totais)} datas) |
| Custodiantes ativos em {ultima:%d/%m/%Y} | {resumo['custodiantes_ativos_ultima_data']} |
| Fundos distintos sob custódia em {ultima:%d/%m/%Y} | {resumo['fundos_distintos_sob_custodia_ultima_data']:,} |
| Pico de custodiantes ativos | {resumo['pico_custodiantes_ativos']} ({resumo['data_pico_custodiantes_ativos']}) |

**Maiores custodiantes em {ultima:%d/%m/%Y}**

{tabela_md(top)}
<!-- RESULTADOS:FIM -->'''
texto = README.read_text(encoding="utf-8")
novo, n = re.subn(r"<!-- RESULTADOS:INICIO -->.*?<!-- RESULTADOS:FIM -->", lambda m: bloco, texto, flags=re.S)
assert n == 1, "marcadores de resultados não encontrados no README"
README.write_text(novo, encoding="utf-8")
print("Arquivos em", SAIDA, ":", sorted(p.name for p in SAIDA.iterdir()))

Arquivos em C:\Users\bcasanovas\Pyhton Files\dash_custodiante_hist\resultados : ['concentracao_por_data.csv', 'custodiantes_serie_historica.csv', 'grafico_custodiantes_ativos.html', 'grafico_fundos_por_custodiante.html', 'grafico_market_share_custodiantes.html', 'qualidade_dados.csv', 'resumo_indicadores.json', 'totais_por_data.csv', 'variacao_custodiantes.csv']
